# Recreating San Antonio's bicycle High Injury Network method

This notebook focuses only on the City's official Bicycle High Injury Network comparison and an independent segment-based recreation. Run the main analysis notebook for trends, city comparisons and victim profiles.

In [ ]:
from pathlib import Path
import os
import zipfile
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
DATA = ROOT / 'data'
RAW = DATA / 'raw'
BOUNDARIES = DATA / 'boundaries'
OUT = ROOT / 'outputs'
BOUNDARIES.mkdir(exist_ok=True)
OUT.mkdir(exist_ok=True)
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 50)

In [ ]:
# Load and filter the CRIS export. The export has 12 metadata rows before the header.
csv_path = RAW / 'myexport_final.csv'
raw = pd.read_csv(csv_path, skiprows=12, low_memory=False)
severity = {'K - FATAL INJURY', 'A - SUSPECTED SERIOUS INJURY'}
target = raw[(raw['Person Type'] == '3 - PEDALCYCLIST') & raw['Person Injury Severity'].isin(severity)].copy()
target['Crash Year'] = pd.to_numeric(target['Crash Year'], errors='coerce').astype('Int64')
target['Latitude_num'] = pd.to_numeric(target['Latitude'], errors='coerce')
target['Longitude_num'] = pd.to_numeric(target['Longitude'], errors='coerce')
target['serious_injury'] = (target['Person Injury Severity'] == 'A - SUSPECTED SERIOUS INJURY').astype(int)
target['death'] = (target['Person Injury Severity'] == 'K - FATAL INJURY').astype(int)
target['affected'] = 1
print('Filtered people:', len(target))
print('Unique crashes:', target['Crash ID'].nunique())
print(target.groupby(['City', 'Crash Year']).agg(people=('affected','sum'), serious_injuries=('serious_injury','sum'), deaths=('death','sum'), crashes=('Crash ID','nunique')).tail(12))

## Official bicycle High Injury Network corridors

This section uses the City's published bicycle HIN corridors and compares the 2019–2023 study period with 2024–Sept. 1, 2026. It counts fatal and suspected serious-injury bicyclist outcomes separately.

In [ ]:
# Download the City's official Bicycle HIN corridor layer.
hin_url = ('https://services.arcgis.com/g1fRTDLeMgspWrYp/arcgis/rest/services/'
           'SS4A_HIN_Dashboard_Data/FeatureServer/1/query?'
           'where=1%3D1&outFields=*&returnGeometry=true&outSR=4326&f=geojson')
hin_file = BOUNDARIES / 'bicycle_hin_corridors.geojson'
if not hin_file.exists():
    response = requests.get(hin_url, timeout=120)
    response.raise_for_status()
    hin_file.write_bytes(response.content)
hin = gpd.read_file(hin_file).to_crs(4326)
hin = hin.rename(columns={'Name': 'corridor'})

# Rebuild one row per qualifying bicyclist crash, including 2026.
hin_people = target[(target['City'] == 'SAN ANTONIO') &
                    target['Crash Year'].between(2019, 2026)].copy()
hin_crashes = (hin_people.groupby('Crash ID', as_index=False)
              .agg(year=('Crash Year', 'first'),
                   latitude=('Latitude_num', 'first'),
                   longitude=('Longitude_num', 'first'),
                   deaths=('death', 'sum'),
                   serious_injuries=('serious_injury', 'sum')))
hin_points = gpd.GeoDataFrame(
    hin_crashes.dropna(subset=['latitude', 'longitude']),
    geometry=gpd.points_from_xy(hin_crashes.dropna(subset=['latitude', 'longitude'])['longitude'],
                                hin_crashes.dropna(subset=['latitude', 'longitude'])['latitude']),
    crs=4326
)

# The buffer makes the matching rule explicit and avoids counting a point
# only when it falls exactly on the centerline geometry.
hin_projected = hin.to_crs(2279).copy()
hin_projected['geometry'] = hin_projected.geometry.buffer(150)
hin_matches = gpd.sjoin(hin_points.to_crs(2279),
                        hin_projected[['bicycle_hin_id', 'corridor', 'Miles', 'geometry']],
                        how='inner', predicate='within')
hin_matches = hin_matches.drop_duplicates(['Crash ID', 'bicycle_hin_id'])

def summarize_hin(frame):
    return (frame.groupby(['bicycle_hin_id', 'corridor', 'Miles'], as_index=False)
            .agg(crashes=('Crash ID', 'nunique'),
                 deaths=('deaths', 'sum'),
                 serious_injuries=('serious_injuries', 'sum'),
                 first_year=('year', 'min'),
                 last_year=('year', 'max'))
            .sort_values(['crashes', 'deaths'], ascending=False))

hin_2019_2023 = summarize_hin(hin_matches[hin_matches['year'].between(2019, 2023)])
hin_2024_2026 = summarize_hin(hin_matches[hin_matches['year'].between(2024, 2026)])
hin_2019_2023.to_csv(OUT / 'bicycle_hin_corridors_2019_2023.csv', index=False)
hin_2024_2026.to_csv(OUT / 'bicycle_hin_corridors_2024_2026.csv', index=False)

# The current reporting-window result needed for the story.
south_general = hin_matches[hin_matches['corridor'].str.upper().eq('S GENERAL MCMULLEN')]
south_general.groupby('year', as_index=False).agg(
    crashes=('Crash ID', 'nunique'), deaths=('deaths', 'sum'),
    serious_injuries=('serious_injuries', 'sum')
)

display(hin_2019_2023)
display(hin_2024_2026)


## Independent HIN-method recreation

The City describes a segment-based ranking with severity weights and a length adjustment. This is an independent recreation, not the City's official calculation. Weights here: fatal crash = 10 points; suspected serious injury = 5 points. Candidate segments are the top 20% by severity score per mile.

In [ ]:
# Use the local City Streets layer included in the repo. This avoids a
# similarly named but geographically incorrect ArcGIS layer.
streets_dir = DATA / 'raw' / 'streets'
street_shp = streets_dir / 'Streets' / 'Streets.shp'
if not street_shp.exists():
    with zipfile.ZipFile(DATA / 'raw' / 'Streets.zip') as archive:
        archive.extractall(streets_dir)
roads = gpd.read_file(street_shp)
roads = roads.rename(columns={'CartID':'segmentid','MSAG_NAME':'road_label','FROM_STREE':'from_street','TO_STREET':'to_street','CoSARoadFu':'road_class'})
roads['length_miles'] = roads['LengthFeet'] / 5280
roads = roads[roads['length_miles'] > 0].copy()
print('Street segments loaded:', len(roads))
print('Street CRS:', roads.crs)


In [ ]:
# Match each qualifying crash to its nearest local street segment.
# The local Streets layer is in feet, so the distance column is feet.
segment_join = gpd.sjoin_nearest(hin_points.to_crs(roads.crs), roads[['segmentid','road_label','from_street','to_street','road_class','length_miles','geometry']], how='left', distance_col='match_distance_ft')
segment_join = segment_join.sort_values('match_distance_ft').drop_duplicates('Crash ID').copy()
segment_join = segment_join[segment_join['match_distance_ft'] <= 150].copy()
print('Crash matches within 150 feet:', len(segment_join))
segment_join['severity_score'] = segment_join['deaths'] * 10 + segment_join['serious_injuries'] * 5

def rank_segments(frame):
    result = (frame.groupby(['segmentid','road_label','from_street','to_street','road_class','length_miles'], dropna=False, as_index=False)
              .agg(crashes=('Crash ID','nunique'), deaths=('deaths','sum'), serious_injuries=('serious_injuries','sum'), severity_score=('severity_score','sum'), first_year=('year','min'), last_year=('year','max'), max_match_distance_ft=('match_distance_ft','max')))
    result['score_per_mile'] = result['severity_score'] / result['length_miles']
    result['candidate_high_injury_segment'] = result['score_per_mile'] >= result['score_per_mile'].quantile(.80)
    return result.sort_values(['candidate_high_injury_segment','score_per_mile','deaths'], ascending=False)

baseline_segments = rank_segments(segment_join[segment_join['year'].between(2019,2023)])
current_segments = rank_segments(segment_join[segment_join['year'].between(2024,2026)])
baseline_segments.to_csv(OUT / 'candidate_hin_segments_2019_2023.csv', index=False)
current_segments.to_csv(OUT / 'candidate_hin_segments_2024_2026.csv', index=False)
display(current_segments.head(20))

# Exploratory variable-length corridor screen. Two crashes are grouped
# only when they are within 500 feet of one another. The 500 feet is a
# proximity rule, not a corridor length: each cluster keeps its observed
# geographic span, like the City's variable-length corridor areas.
from sklearn.cluster import DBSCAN
current_points = segment_join[segment_join['year'].between(2024, 2026)].copy()
xy = np.column_stack([current_points.geometry.x, current_points.geometry.y])
current_points['cluster'] = DBSCAN(eps=500, min_samples=2).fit_predict(xy)
current_points = current_points[current_points['cluster'] >= 0].copy()

def join_values(series):
    values = sorted({str(value).strip() for value in series.dropna() if str(value).strip() and str(value).strip().upper() != 'TBD'})
    return ', '.join(values)

current_clusters = (current_points.groupby('cluster', as_index=False)
    .agg(crashes=('Crash ID', 'nunique'),
         deaths=('deaths', 'sum'),
         serious_injuries=('serious_injuries', 'sum'),
         first_year=('year', 'min'),
         last_year=('year', 'max'),
         roads=('road_label', join_values),
         from_streets=('from_street', join_values),
         to_streets=('to_street', join_values),
         max_match_distance_ft=('match_distance_ft', 'max')))
cluster_span = (current_points.groupby('cluster')['geometry']
    .apply(lambda geom: max(geom.x.max() - geom.x.min(), geom.y.max() - geom.y.min()))
    .rename('corridor_span_ft').reset_index())
current_clusters = current_clusters.merge(cluster_span, on='cluster', how='left')
current_clusters['corridor_span_miles'] = current_clusters['corridor_span_ft'] / 5280
current_clusters = current_clusters.sort_values(
    ['crashes', 'deaths', 'serious_injuries'], ascending=False)
current_clusters.to_csv(OUT / 'bounded_hotspots_2024_2026.csv', index=False)
print('Variable-length repeat-crash corridors:', len(current_clusters))
print('Longest candidate corridor (miles):', round(current_clusters['corridor_span_miles'].max(), 2) if len(current_clusters) else 0)
display(current_clusters)


In [ ]:
# Candidate new trouble spots: current top-20% segments with at least two crashes.
new_candidates = current_segments[(current_segments['candidate_high_injury_segment']) & (current_segments['crashes'] >= 2)].copy()
new_candidates.to_csv(OUT / 'candidate_new_hin_segments_2024_2026.csv', index=False)
display(new_candidates.head(20))


## Reporting-window takeaway

Use `candidate_new_hin_segments_2024_2026.csv` to see which road segments emerge outside the published HIN. These are candidate segments for reporting, not official City corridors. The original HIN comparison above uses the City's published corridor lines and a 150-foot buffer.